In [1]:
# import libraries
from glob import glob
import numpy as np
import scipy.io
import scipy.stats
import torch
import clip
from PIL import Image

In [2]:
# images path
clip.clip.available_models()
images = glob('../stimuli/images/*.png')
images.sort()

In [3]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP model
model, preprocess = clip.load("ViT-L/14@336px", device=device)

all_image_embeddings = []
for artimage in images:
    # Load and preprocess image
    image = preprocess(Image.open(artimage)).unsqueeze(0).to(device)

    # Extract image embedding
    with torch.no_grad():
        image_embedding = model.encode_image(image)
        
    # Normalize (very important for similarity / RSA)
    image_embedding = image_embedding / image_embedding.norm(dim=-1, keepdim=True)

    all_image_embeddings.append(np.array(image_embedding[0,:]))

In [4]:
# save results
CLIPembeddings_similarity_matrix = np.corrcoef(all_image_embeddings)
mdic = {"images_path": images, "CLIP_embeddings": all_image_embeddings,"CLIP_embeddings_similarity_matrix": CLIPembeddings_similarity_matrix}
scipy.io.savemat("CLIP_simi_matrix.mat", mdic)

/Users/liangxinyu/miniforge3/envs/nat/lib/python3.9/site-packages/numpy/lib/function_base.py:495: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis)
/Users/liangxinyu/miniforge3/envs/nat/lib/python3.9/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in true_divide
  ret = um.true_divide(
/Users/liangxinyu/miniforge3/envs/nat/lib/python3.9/site-packages/numpy/lib/function_base.py:2821: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/liangxinyu/miniforge3/envs/nat/lib/python3.9/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)
/Users/liangxinyu/miniforge3/envs/nat/lib/python3.9/site-packages/numpy/lib/function_base.py:2680: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
